# Description

This tool is taken from [langchain-wikibase](https://github.com/donaldziff/langchain-wikibase/tree/main) project. In particular from this [wikibase_agent example](https://github.com/donaldziff/langchain-wikibase/blob/main/wikibase_agent.ipynb)

## Short Description of the Code

This code provides two utility functions for working with Wikidata:

### 🔹 `get_nested_value`
A safe helper function for navigating nested dictionaries and lists.  
It walks through a specified path of keys or indexes and returns the corresponding value.  
If any lookup fails (because of a missing key, wrong type, or out-of-range index), it returns `None` instead of raising an error.

### 🔹 `WikidataEntitySearch`
A lightweight wrapper around the Wikidata Search API.  
It sends a query to Wikidata to find either an **item** (Q-ID) or a **property** (P-ID) that best matches a search term.

**Function features:**
- Supports searching for items (`Qxxxx`) or properties (`Pxxxx`)
- Uses appropriate Wikidata namespaces for each entity type
- Applies relevant search profiles (`srqiprofile`) automatically
- Extracts the top search result safely using `get_nested_value`
- Returns the found Q-ID or P-ID, or a helpful error message if nothing is found

Together, these functions make it easy to perform robust entity lookup queries against Wikidata.

In [2]:
from typing import Optional
import requests

def get_nested_value(o: dict, path: list) -> any:
    """
    Safely walk through nested dicts and lists by keys/indexes.
    Returns None if any KeyError, IndexError, or TypeError occurs.
    """
    current = o
    for key in path:
        try:
            current = current[key]
        except (KeyError, IndexError, TypeError):
            return None
    return current


def WikidataEntitySearch(
    search: str,
    entity_type: str = "item",
    url: str = "https://www.wikidata.org/w/api.php",
    user_agent_header: str = 'DeepWikidataMapper/0.1',
    srqiprofile: str = None,
) -> Optional[str]:
    """
    Search Wikidata entities for a given query.

    Args:
        search: Text to search for (e.g. "Berlin").
        entity_type: Type of entity to search ('item' or 'property').
        url: Wikidata API URL.
        user_agent_header: User-Agent string for requests.
        srqiprofile: Search profile for Wikidata API.

    Returns:
        The Q-ID or P-ID of the top result, or an error message if not found.
    """
    headers = {"Accept": "application/json"}
    if user_agent_header is not None:
        headers["User-Agent"] = user_agent_header

    if entity_type == "item":
        srnamespace = 0
        srqiprofile = "classic_noboostlinks" if srqiprofile is None else srqiprofile
    elif entity_type == "property":
        srnamespace = 120
        srqiprofile = "classic" if srqiprofile is None else srqiprofile
    else:
        raise ValueError("entity_type must be either 'property' or 'item'")

    params = {
        "action": "query",
        "list": "search",
        "srsearch": search,
        "srnamespace": srnamespace,
        "srlimit": 1,
        "srqiprofile": srqiprofile,
        "srwhat": "text",
        "format": "json",
    }

    response = requests.get(url, headers=headers, params=params)

    if response.status_code == 200:
        title = get_nested_value(response.json(), ["query", "search", 0, "title"])
        if title is None:
            return f"I couldn't find any {entity_type} for '{search}'. Please rephrase your request and try again"
        # if there is a prefix, strip it off
        return title.split(":")[-1]
    else:
        return "Sorry, I got an error. Please try again."


# Function test

In [3]:
WikidataEntitySearch('Extended Semantic Web Conference')

'Q17012957'